<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;"><div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div><div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div></div>
<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;"><h1 style="margin:0;"><b>Image Processing Fundamentals — Algorithm</b></h1></div>


## Purpose and reading order

This notebook is the algorithm specification for the corresponding `main.ipynb` in this lab. It has two linked layers:

| Layer | Content | Question answered |
| --- | --- | --- |
| **Part I — Complete pseudocode** | One self-contained executable-style block with explicit branches and gates | What is the whole pipeline? |
| **Part II — Auditable specification** | Six-section contract for every step: Purpose, Inputs, Procedure, Produces, Acceptance, Failure mode | Why is each step valid? |

Names, shapes, thresholds, and failure decisions are kept identical between both layers. The acceptance contract of step *k* is the precondition of step *k+1*.

## Part I — Complete Pseudocode

The single block below is the complete pipeline. Every named artifact is assigned before use; every failure is explicit. It is a direct, implementation-neutral transcription of the executable notebook.

```text
ALGORITHM Image_Processing_Fundamentals

# Algorithm 1 — Setup — Environment and Configuration
RNG ← ARRAY.random.default_rng(42)
PLOT.rcParams["figure.dpi"] ← 110
PLOT.rcParams["axes.titlesize"] ← 11

# Algorithm 2 — 1. Data and Output Paths
PROCEDURE locate_lab_root(start)
    start ← start.resolve()
    FOR candidate IN [start, EXPAND start.parents] DO
        IF ((candidate / "data").is_dir() AND (candidate / "notebooks").is_dir()) THEN
            RETURN candidate
        END IF
    END FOR
    RAISE FileNotFoundError("Could not locate the lab root containing data/ and notebooks/.")
END PROCEDURE
LAB_DIR ← locate_lab_root(PATH.cwd())
DATA_DIR ← (LAB_DIR / "data")
OUTPUT_DIR ← ((LAB_DIR / "outputs") / "figures")
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
IMAGE_EXTENSIONS ← {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATA_FILES ← SORT(STREAM[path FOR path IN DATA_DIR.rglob("*") WHERE (path.is_file() AND (path.suffix.lower() IN IMAGE_EXTENSIONS))])
REQUIRE DATA_FILES OTHERWISE FORMAT("No supported images found in: {0}" , DATA_DIR)
PROCEDURE select_input_image(VARIADIC keywords)
    keywords ← TUPLE(STREAM[keyword.lower() FOR keyword IN keywords])
    matches ← LIST[path FOR path IN DATA_FILES WHERE ALL(STREAM[(keyword IN path.stem.lower()) FOR keyword IN keywords])]
    IF (NOT matches) THEN
        RAISE FileNotFoundError(FORMAT("No image matching {0} found in {1}" , keywords, DATA_DIR))
    END IF
    RETURN matches[0]
END PROCEDURE
ROLE_PATTERNS ← {"einstein" ↦ ("einstein",), "peppers" ↦ ("pepper",), "ballons" ↦ ("ballon",), "grass" ↦ ("grass",
    ), "tower" ↦ ("tower",)}
IMAGE_FILES ← MAP{role ↦ select_input_image(EXPAND patterns) FOR (role, patterns) IN ITEMS(ROLE_PATTERNS)}
CALL EMIT("Lab directory :", LAB_DIR)
CALL EMIT("Data directory:", DATA_DIR)
CALL EMIT("Output folder :", OUTPUT_DIR)
CALL EMIT("Images discovered:", LENGTH(DATA_FILES))
CALL EMIT("Experiment roles :", LENGTH(IMAGE_FILES))

# Algorithm 3 — Sampling Experiment
x ← ARRAY.linspace(0, (2 * ARRAY.pi), 256)
y ← ARRAY.linspace(0, (2 * ARRAY.pi), 256)
(xx, yy) ← ARRAY.meshgrid(x, y)
continuous_signal_like ← ((0.55 + (0.25 * ARRAY.sin((2.0 * xx)))) + (0.2 * ARRAY.cos((3.0 * yy))))
continuous_signal_like ← ARRAY.clip(continuous_signal_like, 0.0, 1.0)
sampling_intervals ← [1, 4, 8, 16]
(fig, axes) ← PLOT.subplots(1, 4, figsize = (14, 3.4))
FOR (ax, step) IN ZIP(axes, sampling_intervals) DO
    sampled_signal ← continuous_signal_like[(::step, ::step)]
    CALL ax.imshow(sampled_signal, cmap = "gray", vmin = 0, vmax = 1, interpolation = "nearest")
    CALL ax.set_title(FORMAT("Sampling step = {0}\nshape = {1}" , step, sampled_signal.shape))
    CALL ax.axis("off")
END FOR
CALL fig.suptitle("Spatial Sampling")
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "01_sampling.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 4 — Quantization Experiment
gradient ← ARRAY.tile(ARRAY.linspace(0, 1, 512), (90, 1))
quantization_bit_depths ← [1, 2, 4, 8]
(fig, axes) ← PLOT.subplots(4, 1, figsize = (11, 6))
FOR (ax, bits) IN ZIP(axes, quantization_bit_depths) DO
    levels ← (2 ^ bits)
    quantized ← (ARRAY.round((gradient * (levels − 1))) / (levels − 1))
    CALL ax.imshow(quantized, cmap = "gray", vmin = 0, vmax = 1, aspect = "auto")
    CALL ax.set_title(FORMAT("{0}-bit quantization → {1} intensity levels" , bits, levels))
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "02_quantization.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 5 — 3. Pixel Coordinates and Array Representation
toy_grayscale_image ← ARRAY.array([[0, 32, 64, 96, 128], [24, 56, 88, 120, 152], [48, 80, 112, 144, 176],
    [72, 104, 136, 168, 208], [96, 128, 160, 208, 255]], dtype = ARRAY.uint8)
(fig, ax) ← PLOT.subplots(figsize = (5, 4.5))
CALL ax.imshow(toy_grayscale_image, cmap = "gray", vmin = 0, vmax = 255)
FOR row IN RANGE(toy_grayscale_image.shape[0]) DO
    FOR col IN RANGE(toy_grayscale_image.shape[1]) DO
        CALL ax.text(col, row, STRING(toy_grayscale_image[(row, col)]), ha = "center", va = "center",
            fontsize = 8)
    END FOR
END FOR
CALL ax.set_title("A grayscale image is a matrix of intensities")
CALL ax.set_xlabel("x / column")
CALL ax.set_ylabel("y / row")
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "03_grayscale_matrix.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
CALL EMIT("Shape:", toy_grayscale_image.shape)
CALL EMIT("dtype:", toy_grayscale_image.dtype)
CALL EMIT("Pixel at row=2, column=3:", toy_grayscale_image[(2, 3)])

# Algorithm 6 — 4. Image Representation Modes
binary ← ARRAY.zeros((120, 160), dtype = ARRAY.uint8)
binary[(30:90, 45:120)] ← 255
grayscale ← ARRAY.tile(ARRAY.linspace(0, 255, 160, dtype = ARRAY.uint8), (120, 1))
rgb ← ARRAY.zeros((120, 160, 3), dtype = ARRAY.uint8)
rgb[(:, :53)] ← [255, 0, 0]
rgb[(:, 53:106)] ← [0, 255, 0]
rgb[(:, 106:)] ← [0, 0, 255]
(fig, axes) ← PLOT.subplots(1, 3, figsize = (11, 3.3))
CALL axes[0].imshow(binary, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title(FORMAT("Binary\nshape={0}" , binary.shape))
CALL axes[1].imshow(grayscale, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title(FORMAT("Grayscale\nshape={0}" , grayscale.shape))
CALL axes[2].imshow(rgb)
CALL axes[2].set_title(FORMAT("RGB\nshape={0}" , rgb.shape))
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "04_image_types.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 7 — 5. Load and Inspect Real Images
images ← {}
FOR (name, path) IN ITEMS(IMAGE_FILES) DO
    images[name] ← ARRAY.asarray(IMAGE.open(path).convert("RGB"))
END FOR
FOR (name, image) IN ITEMS(images) DO
    CALL EMIT(FORMAT("{0:9s} | shape={1:16s} dtype={2} range=[{3}, {4}]" , name, STRING(image.shape),
        image.dtype, MIN(image), MAX(image)))
END FOR
(fig, axes) ← PLOT.subplots(1, LENGTH(images), figsize = (16, 4))
FOR (ax, (name, image)) IN ZIP(axes, ITEMS(images)) DO
    CALL ax.imshow(image)
    CALL ax.set_title(FORMAT("{0}\n{1}×{2}" , name, image.shape[1], image.shape[0]))
    CALL ax.axis("off")
END FOR
CALL fig.suptitle("Reference Images")
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "05_reference_images.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 8 — 6. Dimensions, Resolution, Aspect Ratio, and Channels
peppers_image ← images["peppers"]
(height, width, channels) ← peppers_image.shape
pixel_count ← (height * width)
aspect_ratio ← (width / height)
CALL EMIT(FORMAT("Height       : {0} pixels" , height))
CALL EMIT(FORMAT("Width        : {0} pixels" , width))
CALL EMIT(FORMAT("Channels     : {0}" , channels))
CALL EMIT(FORMAT("Pixel count  : {0:,}" , pixel_count))
CALL EMIT(FORMAT("Aspect ratio : {0:.3f}" , aspect_ratio))

# Algorithm 9 — 7. Data Types, Bit Depth, Dynamic Range, and Memory
CALL EMIT("dtype:", peppers_image.dtype)
CALL EMIT("bytes per value:", peppers_image.dtype.itemsize)
CALL EMIT("array memory:", FORMAT("{0:,} bytes" , peppers_image.nbytes))
CALL EMIT("array memory:", FORMAT("{0:.3f} MiB" , (peppers_image.nbytes / (1024 ^ 2))))
uint8_info ← ARRAY.iinfo(ARRAY.uint8)
CALL EMIT("uint8 range:", uint8_info.min, "to", uint8_info.max)

# Algorithm 10 — 8. Display Scaling and Visualization Control
low_contrast ← ARRAY.linspace(90, 165, 256, dtype = ARRAY.uint8)
low_contrast ← ARRAY.tile(low_contrast, (120, 1))
(fig, axes) ← PLOT.subplots(1, 2, figsize = (9, 3.2))
CALL axes[0].imshow(low_contrast, cmap = "gray")
CALL axes[0].set_title("Automatic display scaling")
CALL axes[1].imshow(low_contrast, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Fixed display range: 0–255")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "06_display_scaling.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 11 — 9. Pixel Access and Safe Modification
balloons_image ← images["ballons"]
(y, x) ← (140, 220)
original_pixel ← COPY(balloons_image[(y, x)])
edited_balloons_image ← COPY(balloons_image)
radius ← 6
edited_balloons_image[((y − radius):((y + radius) + 1), (x − radius):((x + radius) + 1))] ← [255, 0, 255]
(fig, axes) ← PLOT.subplots(1, 2, figsize = (10, 4))
CALL axes[0].imshow(balloons_image)
CALL axes[0].scatter([x], [y], s = 70, facecolors = "none", edgecolors = "yellow")
CALL axes[0].set_title("Original + selected pixel")
CALL axes[1].imshow(edited_balloons_image)
CALL axes[1].set_title("Edited copy")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "07_pixel_edit.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
CALL EMIT("Selected coordinate (x, y):", (x, y))
CALL EMIT("Stored RGB value:", original_pixel)

# Algorithm 12 — 10. Regions of Interest (ROI)
tower_image ← images["tower"]
(y0, y1) ← (120, 360)
(x0, x1) ← (170, 390)
region_of_interest ← tower_image[(y0:y1, x0:x1)]
(fig, axes) ← PLOT.subplots(1, 2, figsize = (10, 4.5))
CALL axes[0].imshow(tower_image)
CALL axes[0].add_patch(PLOT.Rectangle((x0, y0), (x1 − x0), (y1 − y0), fill = FALSE, edgecolor = "red",
    linewidth = 2))
CALL axes[0].set_title("Full image and ROI")
CALL axes[1].imshow(region_of_interest)
CALL axes[1].set_title(FORMAT("ROI: {0}×{1}" , region_of_interest.shape[1], region_of_interest.shape[0]))
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "08_region_of_interest.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 13 — 11. Pixel Neighborhoods
einstein_rgb ← images["einstein"]
einstein_grayscale_float ← (((0.299 * CAST(einstein_rgb[(…, 0)],
    ARRAY.float32)) + (0.587 * CAST(einstein_rgb[(…, 1)], ARRAY.float32))) + (0.114 * CAST(einstein_rgb[(…,
    2)], ARRAY.float32)))
einstein_gray ← CAST(ARRAY.clip(einstein_grayscale_float, 0, 255), ARRAY.uint8)
(y, x) ← (120, 120)
center_patch_3x3 ← einstein_gray[((y − 1):(y + 2), (x − 1):(x + 2))]
CALL EMIT("Center pixel:", einstein_gray[(y, x)])
CALL EMIT("3×3 neighborhood:")
CALL EMIT(center_patch_3x3)

# Algorithm 14 — 12. RGB Channel Decomposition
red ← peppers_image[(…, 0)]
green ← peppers_image[(…, 1)]
blue ← peppers_image[(…, 2)]
(fig, axes) ← PLOT.subplots(1, 4, figsize = (14, 4))
CALL axes[0].imshow(peppers_image)
CALL axes[0].set_title("RGB")
CALL axes[1].imshow(red, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Red channel")
CALL axes[2].imshow(green, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[2].set_title("Green channel")
CALL axes[3].imshow(blue, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[3].set_title("Blue channel")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "09_rgb_channels.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
CALL EMIT("Channel means:", {"R" ↦ ROUND(REAL(MEAN(red)), 2), "G" ↦ ROUND(REAL(MEAN(green)), 2),
    "B" ↦ ROUND(REAL(MEAN(blue)), 2)})

# Algorithm 15 — 13. RGB and BGR Conventions
rgb_pixel_example ← peppers_image
bgr_channel_order_example ← rgb_pixel_example[(…, ::(−1))]
(fig, axes) ← PLOT.subplots(1, 2, figsize = (9, 4))
CALL axes[0].imshow(rgb_pixel_example)
CALL axes[0].set_title("Correct RGB")
CALL axes[1].imshow(bgr_channel_order_example)
CALL axes[1].set_title("Channels reversed")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "10_rgb_bgr.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 16 — 14. RGB-to-Grayscale Conversion
PROCEDURE convert_rgb_to_grayscale(rgb_image)
    rgb_float ← CAST(rgb_image, ARRAY.float32)
    gray ← (((0.299 * rgb_float[(…, 0)]) + (0.587 * rgb_float[(…, 1)])) + (0.114 * rgb_float[(…, 2)]))
    RETURN CAST(ARRAY.clip(gray, 0, 255), ARRAY.uint8)
END PROCEDURE
einstein_rgb ← images["einstein"]
einstein_gray ← convert_rgb_to_grayscale(einstein_rgb)
(fig, axes) ← PLOT.subplots(1, 2, figsize = (9, 4))
CALL axes[0].imshow(einstein_rgb)
CALL axes[0].set_title(FORMAT("RGB shape: {0}" , einstein_rgb.shape))
CALL axes[1].imshow(einstein_gray, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title(FORMAT("Grayscale shape: {0}" , einstein_gray.shape))
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "11_rgb_to_grayscale.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 17 — 15. Image Statistics
grass_grayscale_image ← convert_rgb_to_grayscale(images["grass"])
statistics ← {"min" ↦ INTEGER(MIN(grass_grayscale_image)), "max" ↦ INTEGER(MAX(grass_grayscale_image)),
    "mean" ↦ REAL(MEAN(grass_grayscale_image)), "median" ↦ REAL(ARRAY.median(grass_grayscale_image)),
    "std" ↦ REAL(STD(grass_grayscale_image)), "p05" ↦ REAL(ARRAY.percentile(grass_grayscale_image, 5)),
    "p95" ↦ REAL(ARRAY.percentile(grass_grayscale_image, 95))}
FOR (name, value) IN ITEMS(statistics) DO
    CALL EMIT((IF IS_INSTANCE(value, REAL) THEN FORMAT("{0:>6s}: {1:.3f}" , name,
        value) ELSE FORMAT("{0:>6s}: {1}" , name, value)))
END FOR

# Algorithm 18 — 16. Intensity Histograms
balloons_grayscale_image ← convert_rgb_to_grayscale(images["ballons"])
(counts, bin_edges) ← ARRAY.histogram(FLATTEN(balloons_grayscale_image), bins = 256, range = (0, 256))
(fig, axes) ← PLOT.subplots(1, 2, figsize = (11, 4))
CALL axes[0].imshow(balloons_grayscale_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Grayscale image")
CALL axes[0].axis("off")
CALL axes[1].plot(ARRAY.arange(256), counts)
CALL axes[1].set_title("Intensity histogram")
CALL axes[1].set_xlabel("Intensity")
CALL axes[1].set_ylabel("Pixel count")
CALL axes[1].set_xlim(0, 255)
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "12_intensity_histogram.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
CALL EMIT("Histogram count:", SUM(counts))
CALL EMIT("Number of image pixels:", balloons_grayscale_image.size)
shuffled_pixel_image ← COPY(FLATTEN(balloons_grayscale_image))
CALL RNG.shuffle(shuffled_pixel_image)
shuffled_pixel_image ← RESHAPE(shuffled_pixel_image, balloons_grayscale_image.shape)
(hist_original, _) ← ARRAY.histogram(FLATTEN(balloons_grayscale_image), bins = 256, range = (0, 256))
(hist_shuffled, _) ← ARRAY.histogram(FLATTEN(shuffled_pixel_image), bins = 256, range = (0, 256))
CALL EMIT("Histograms identical:", ARRAY.array_equal(hist_original, hist_shuffled))
(fig, axes) ← PLOT.subplots(1, 2, figsize = (9, 4))
CALL axes[0].imshow(balloons_grayscale_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Original")
CALL axes[1].imshow(shuffled_pixel_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Pixels shuffled")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL PLOT.tight_layout()
CALL PLOT.show()

# Algorithm 19 — 17. Dynamic Range and Min-Max Normalization
PROCEDURE normalize_minmax(gray_image)
    image_float ← CAST(gray_image, ARRAY.float32)
    minimum ← MIN(image_float)
    maximum ← MAX(image_float)
    IF (maximum = minimum) THEN
        RETURN ARRAY.zeros_like(gray_image)
    END IF
    normalized ← ((image_float − minimum) / (maximum − minimum))
    normalized ← (normalized * 255.0)
    RETURN CAST(ARRAY.clip(normalized, 0, 255), ARRAY.uint8)
END PROCEDURE
source_grayscale_image ← convert_rgb_to_grayscale(images["ballons"])
low_contrast ← (90 + ((CAST(source_grayscale_image, ARRAY.float32) / 255.0) * 76))
low_contrast ← CAST(ARRAY.clip(low_contrast, 0, 255), ARRAY.uint8)
normalized ← normalize_minmax(low_contrast)
(fig, axes) ← PLOT.subplots(2, 2, figsize = (10, 7))
CALL axes[(0, 0)].imshow(low_contrast, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[(0, 0)].set_title("Low-contrast image")
CALL axes[(0, 0)].axis("off")
CALL axes[(0, 1)].hist(FLATTEN(low_contrast), bins = 256, range = (0, 256))
CALL axes[(0, 1)].set_title("Before normalization")
CALL axes[(0, 1)].set_xlabel("Intensity")
CALL axes[(1, 0)].imshow(normalized, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[(1, 0)].set_title("After min-max normalization")
CALL axes[(1, 0)].axis("off")
CALL axes[(1, 1)].hist(FLATTEN(normalized), bins = 256, range = (0, 256))
CALL axes[(1, 1)].set_title("After normalization")
CALL axes[(1, 1)].set_xlabel("Intensity")
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "13_dynamic_range_normalization.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
CALL EMIT("Before range:", INTEGER(MIN(low_contrast)), "to", INTEGER(MAX(low_contrast)))
CALL EMIT("After range :", INTEGER(MIN(normalized)), "to", INTEGER(MAX(normalized)))

# Algorithm 20 — 18. `uint8` Arithmetic, Overflow, Clipping, and Floating Point
value ← ARRAY.array([250], dtype = ARRAY.uint8)
unsafe ← (value + ARRAY.array([20], dtype = ARRAY.uint8))
safe_float ← (CAST(value, ARRAY.float32) + 20.0)
safe_uint8 ← CAST(ARRAY.clip(safe_float, 0, 255), ARRAY.uint8)
CALL EMIT("Original value :", value[0])
CALL EMIT("Unsafe result  :", unsafe[0])
CALL EMIT("Safe result    :", safe_uint8[0])

# Algorithm 21 — 19. Noise Model Simulation
base_grayscale_image ← einstein_gray
gaussian_noise_samples ← RNG.normal(0.0, 20.0, size = base_grayscale_image.shape)
gaussian_noisy_image ← CAST(ARRAY.clip((CAST(base_grayscale_image, ARRAY.float32) + gaussian_noise_samples),
    0, 255), ARRAY.uint8)
salt_and_pepper_noisy_image ← COPY(base_grayscale_image)
probability ← 0.03
salt_pepper_random_map ← RNG.random(base_grayscale_image.shape)
salt_and_pepper_noisy_image[(salt_pepper_random_map < (probability / 2))] ← 0
salt_and_pepper_noisy_image[(salt_pepper_random_map > (1 − (probability / 2)))] ← 255
scaled ← (CAST(base_grayscale_image, ARRAY.float32) / 255.0)
poisson_noisy_image ← (RNG.poisson((scaled * 30.0)) / 30.0)
poisson_noisy_image ← CAST(ARRAY.clip((poisson_noisy_image * 255.0), 0, 255), ARRAY.uint8)
speckle_noise ← RNG.normal(0.0, 0.18, size = base_grayscale_image.shape)
speckle_noisy_image ← (CAST(base_grayscale_image, ARRAY.float32) * (1.0 + speckle_noise))
speckle_noisy_image ← CAST(ARRAY.clip(speckle_noisy_image, 0, 255), ARRAY.uint8)
(fig, axes) ← PLOT.subplots(1, 5, figsize = (16, 3.6))
examples ← [("Original", base_grayscale_image), ("Gaussian", gaussian_noisy_image), ("Salt & pepper",
    salt_and_pepper_noisy_image), ("Poisson", poisson_noisy_image), ("Speckle", speckle_noisy_image)]
FOR (ax, (title, image)) IN ZIP(axes, examples) DO
    CALL ax.imshow(image, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "14_noise_models.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 22 — 20. Image Comparison Metrics
PROCEDURE compute_image_quality_metrics(reference, test)
    IF (reference.shape ≠ test.shape) THEN
        RAISE ValueError("Images must have identical shapes.")
    END IF
    reference_f ← CAST(reference, ARRAY.float64)
    test_f ← CAST(test, ARRAY.float64)
    difference ← (reference_f − test_f)
    mae ← MEAN(ARRAY, ARRAY.abs(difference))
    mse ← MEAN(ARRAY, (difference ^ 2))
    rmse ← ARRAY.sqrt(mse)
    IF (mse = 0) THEN
        psnr ← ARRAY.inf
    ELSE
        psnr ← (10.0 * ARRAY.log10(((255.0 ^ 2) / mse)))
    END IF
    RETURN {"MAE" ↦ mae, "MSE" ↦ mse, "RMSE" ↦ rmse, "PSNR" ↦ psnr}
END PROCEDURE
noise_results ← {"Gaussian" ↦ compute_image_quality_metrics(base_grayscale_image, gaussian_noisy_image),
    "Salt & pepper" ↦ compute_image_quality_metrics(base_grayscale_image, salt_and_pepper_noisy_image),
    "Poisson" ↦ compute_image_quality_metrics(base_grayscale_image, poisson_noisy_image),
    "Speckle" ↦ compute_image_quality_metrics(base_grayscale_image, speckle_noisy_image)}
FOR (noise_name, metrics) IN ITEMS(noise_results) DO
    CALL EMIT(noise_name)
    FOR (metric_name, value) IN ITEMS(metrics) DO
        CALL EMIT(FORMAT("  {0:>4s}: {1:.4f}" , metric_name, value))
    END FOR
END FOR

# Algorithm 23 — 21. Lossless vs Lossy Image Encoding
example ← images["peppers"]
png_path ← (OUTPUT_DIR / "15_saved_example.png")
jpg_path ← (OUTPUT_DIR / "15_saved_example.jpg")
CALL IMAGE.fromarray(example).save(png_path)
CALL IMAGE.fromarray(example).save(jpg_path, quality = 75)
png_reload ← ARRAY.asarray(IMAGE.open(png_path).convert("RGB"))
jpg_reload ← ARRAY.asarray(IMAGE.open(jpg_path).convert("RGB"))
png_metrics ← compute_image_quality_metrics(example, png_reload)
jpg_metrics ← compute_image_quality_metrics(example, jpg_reload)
CALL EMIT("PNG reload MSE :", png_metrics["MSE"])
CALL EMIT("JPEG reload MSE:", jpg_metrics["MSE"])
CALL EMIT("PNG path :", png_path)
CALL EMIT("JPEG path:", jpg_path)

# Algorithm 24 — 22. Standard Image Inspection Workflow
PROCEDURE summarize_image_properties(name, image)
    CALL EMIT(FORMAT("Name       : {0}" , name))
    CALL EMIT(FORMAT("Shape      : {0}" , image.shape))
    CALL EMIT(FORMAT("Dimensions : {0}" , image.ndim))
    CALL EMIT(FORMAT("dtype      : {0}" , image.dtype))
    CALL EMIT(FORMAT("Min / max  : {0} / {1}" , MIN(image), MAX(image)))
    CALL EMIT(FORMAT("Mean       : {0:.3f}" , MEAN(image)))
    CALL EMIT(FORMAT("Memory     : {0:,} bytes" , image.nbytes))
END PROCEDURE
CALL summarize_image_properties("peppers", peppers_image)

# Algorithm 25 — 23. Validation Checks
REQUIRE (peppers_image.ndim = 3) OTHERWISE STOP WITH ValidationError
REQUIRE (peppers_image.shape[2] = 3) OTHERWISE STOP WITH ValidationError
REQUIRE (einstein_gray.ndim = 2) OTHERWISE STOP WITH ValidationError
REQUIRE (peppers_image.dtype = ARRAY.uint8) OTHERWISE STOP WITH ValidationError
REQUIRE (einstein_gray.dtype = ARRAY.uint8) OTHERWISE STOP WITH ValidationError
REQUIRE (SUM(counts) = balloons_grayscale_image.size) OTHERWISE STOP WITH ValidationError
REQUIRE (MIN(normalized) = 0) OTHERWISE STOP WITH ValidationError
REQUIRE (MAX(normalized) = 255) OTHERWISE STOP WITH ValidationError
self_metrics ← compute_image_quality_metrics(einstein_gray, einstein_gray)
REQUIRE (self_metrics["MAE"] = 0) OTHERWISE STOP WITH ValidationError
REQUIRE (self_metrics["MSE"] = 0) OTHERWISE STOP WITH ValidationError
REQUIRE (self_metrics["RMSE"] = 0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isinf(self_metrics["PSNR"]) OTHERWISE STOP WITH ValidationError
REQUIRE (region_of_interest.shape[0] = (y1 − y0)) OTHERWISE STOP WITH ValidationError
REQUIRE (region_of_interest.shape[1] = (x1 − x0)) OTHERWISE STOP WITH ValidationError
CALL EMIT("All fundamental validation checks passed.")
REQUIRED_OUTPUTS ← ["01_sampling.png", "02_quantization.png", "03_grayscale_matrix.png", "04_image_types.png",
    "05_reference_images.png", "06_display_scaling.png", "07_pixel_edit.png", "08_region_of_interest.png",
    "09_rgb_channels.png", "10_rgb_bgr.png", "11_rgb_to_grayscale.png", "12_intensity_histogram.png", "13_dynamic_range_normalization.png",
    "14_noise_models.png", "15_saved_example.png", "15_saved_example.jpg"]
missing_outputs ← LIST[output_name FOR output_name IN REQUIRED_OUTPUTS WHERE (NOT (OUTPUT_DIR / output_name).exists())]
IF missing_outputs THEN
    RAISE FileNotFoundError(("Missing outputs: " + ", ".join(missing_outputs)))
END IF
CALL EMIT(FORMAT("Output validation passed: {0} files." , LENGTH(REQUIRED_OUTPUTS)))

END ALGORITHM
```

## Part II — Step-by-step elaboration

Each step below is a six-section contract corresponding to the numbered block in Part I:

- **Purpose** — why the step exists.
- **Inputs** — artifacts and shapes required before execution.
- **Procedure** — ordered computation.
- **Produces** — outputs and invariants.
- **Acceptance contract** — numerical or structural gates before the next step.
- **Failure mode** — explicit abort, skip, or diagnostic action; never a silent substitution.

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project environment in VS Code.
2. Use this algorithm as the coding checklist for `main.ipynb`.
3. Keep synthetic demonstrations separate from real-image inspection so each concept remains isolated.

**Produces**

Selected environment and a top-to-bottom execution plan.

**Acceptance contract**

Review the local assumptions before using cached notebook outputs.

**Failure mode**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create Setup and Discover Data

**Purpose**

Make input selection and noise reproducible.

**Inputs**

Lab directory and image files.

**Procedure**

1. Import `Path`, NumPy, Matplotlib, and PIL.
2. Create one fixed NumPy random generator for reproducible noise.
3. Locate the lab root and create data/output paths.
4. Discover supported images and map Einstein, peppers, balloons, grass, and tower to deterministic files.

**Produces**

Five image roles; RNG seeded with 42; output path.

**Acceptance contract**

All roles resolve to readable inputs.

**Failure mode**

Stop on missing roles or dependencies.

## 2. Demonstrate Spatial Sampling

**Purpose**

Separate spatial resolution from intensity precision.

**Inputs**

Synthetic 2-D signal and positive sampling strides.

**Procedure**

1. Create a smooth 2-D synthetic signal from known sinusoidal components.
2. Choose several sampling intervals.
3. For each interval subsample by array slicing without interpolation.
4. Display each sampled array with its resulting shape and save the comparison.

**Produces**

Subsampled arrays and comparison figure.

**Acceptance contract**

Array dimensions follow the slicing stride; no interpolation is applied.

**Failure mode**

Correct invalid strides; discuss aliasing rather than interpreting it as noise.

## 3. Demonstrate Intensity Quantization

**Purpose**

Isolate the effect of bit depth.

**Inputs**

Normalized gradient and chosen bit depths.

**Procedure**

1. Create a continuous grayscale gradient.
2. For each bit depth compute `levels = 2**bits`.
3. Map the normalized gradient to exactly that number of intensity levels.
4. Display the quantized results with a common intensity range.

**Produces**

Quantized gradients with at most 2**bits levels.

**Acceptance contract**

Values remain in the display range; spatial dimensions stay fixed.

**Failure mode**

Correct rounding/range errors before comparing banding.

## 4. Represent Pixels as Arrays

**Purpose**

Connect image coordinates with array indexing.

**Inputs**

Small known grayscale matrix.

**Procedure**

1. Create a small toy grayscale matrix with explicit values.
2. Print matrix, shape, dtype, and selected pixel values.
3. Use `[row,column] = [y,x]` consistently when translating image coordinates to NumPy indexing.

**Produces**

Pixel values, shape and coordinate illustration.

**Acceptance contract**

NumPy uses [row,column]=[y,x]; selected indices are in bounds.

**Failure mode**

Correct swapped coordinates before extending to real images.

## 5. Create Binary, Grayscale, and RGB Representations

**Purpose**

Distinguish channels from spatial dimensions.

**Inputs**

Synthetic binary, grayscale and RGB arrays.

**Procedure**

1. Construct one binary image, one grayscale gradient, and one RGB image.
2. Verify their shapes distinguish 2-D single-channel from 3-D color arrays.
3. Display side by side and save the representation figure.

**Produces**

Representation comparison figure.

**Acceptance contract**

Single-channel arrays are 2-D; RGB is (height,width,3).

**Failure mode**

Correct channel layout or display mode.

## 6. Load and Inspect Real Images

**Purpose**

Confirm the selected files and image conventions.

**Inputs**

Five resolved image paths.

**Procedure**

1. Load all semantic-role files as RGB arrays.
2. Print shape, dtype, and intensity range for each image.
3. Create a reference-image figure to verify the correct files were selected.

**Produces**

RGB arrays and reference overview.

**Acceptance contract**

Nonempty arrays; uint8 values in [0,255]; expected visual subjects.

**Failure mode**

Stop on decoding errors or incorrect role matching.

## 7. Inspect Resolution, Aspect Ratio, Channels, and Memory

**Purpose**

Measure storage and geometry correctly.

**Inputs**

Loaded image shapes and dtype.

**Procedure**

1. For representative images compute height, width, channel count, and aspect ratio.
2. Read dtype and item size to infer bit depth.
3. Use `nbytes` to report array memory.
4. Keep array memory distinct from compressed file size.

**Produces**

Height, width, aspect ratio, channels and memory summaries.

**Acceptance contract**

Aspect ratio is width/height; nbytes equals size*itemsize.

**Failure mode**

Do not equate compressed file size with array memory.

## 8. Control Display Scaling

**Purpose**

Reveal the effect of visualization scaling.

**Inputs**

One low-contrast image.

**Procedure**

1. Create or select a low-contrast image.
2. Display once with automatic scaling and once with fixed `vmin=0,vmax=255`.
3. Use this to show display settings can change appearance without changing pixels.

**Produces**

Automatic-scale and fixed-scale displays.

**Acceptance contract**

Both panels use identical pixel data; fixed limits are 0 and 255.

**Failure mode**

Correct display settings before claiming an image was enhanced.

## 9. Access and Modify Pixels Safely

**Purpose**

Modify a local region without overwriting the reference.

**Inputs**

RGB array and chosen pixel coordinates.

**Procedure**

1. Choose one `(x,y)` coordinate and read RGB with `[y,x]`.
2. Copy the source before editing.
3. Modify only a small neighborhood in the copy.
4. Display original selection and edited copy so the source remains intact.

**Produces**

Edited copy and comparison.

**Acceptance contract**

The original is unchanged; modified indices are in bounds.

**Failure mode**

Create an independent copy if edits leak into the source.

## 10. Extract a Region of Interest

**Purpose**

Make crop coordinates auditable.

**Inputs**

Image and y0:y1, x0:x1 bounds.

**Procedure**

1. Define explicit `y0:y1` and `x0:x1` bounds.
2. Slice the image as `[y0:y1, x0:x1]`.
3. Draw the same rectangle on the full image.
4. Verify ROI dimensions equal the coordinate differences.

**Produces**

ROI and rectangle on the full image.

**Acceptance contract**

ROI height=y1-y0; width=x1-x0; bounds inside source.

**Failure mode**

Fix bounds rather than accepting an empty or truncated crop.

## 11. Inspect Pixel Neighborhoods

**Purpose**

Inspect local pixel context.

**Inputs**

An interior pixel and neighborhood radius.

**Procedure**

1. Select a center pixel away from the border.
2. Extract a local neighborhood around it.
3. Print or display neighborhood values.
4. Use bounds that prevent unintended negative or wrapped slicing.

**Produces**

Neighborhood values.

**Acceptance contract**

Window fits inside the image and includes the intended centre.

**Failure mode**

Move the pixel or document border handling; avoid negative slicing.

## 12. Decompose RGB Channels

**Purpose**

Relate color appearance to channel values.

**Inputs**

RGB image.

**Procedure**

1. Extract red, green, and blue planes.
2. Display original plus each channel with common grayscale limits.
3. Compute simple per-channel summaries such as mean intensity.

**Produces**

R, G and B planes and summaries.

**Acceptance contract**

Each channel has source spatial shape; common grayscale limits.

**Failure mode**

Correct axis selection if channels are spatial slices instead.

## 13. Demonstrate RGB versus BGR Ordering

**Purpose**

Show the consequence of channel-order mismatch.

**Inputs**

Known RGB array.

**Procedure**

1. Create a channel-reversed copy of one RGB image.
2. Display correct RGB and reversed-channel versions.
3. Use this as the coding reminder that OpenCV and PIL/Matplotlib use different channel conventions.

**Produces**

Channel-reversed example.

**Acceptance contract**

Only channel order changes; reversing twice restores RGB.

**Failure mode**

Use a conversion at the library boundary, not a new color interpretation.

## 14. Convert RGB to Grayscale

**Purpose**

Compute luminance-like grayscale values safely.

**Inputs**

RGB uint8 array.

**Procedure**

1. Convert RGB to float before weighted arithmetic.
2. Compute `0.299R + 0.587G + 0.114B`.
3. Clip to `[0,255]` and cast to uint8.
4. Verify the output is 2-D and preserves source height/width.

**Produces**

2-D uint8 grayscale image.

**Acceptance contract**

Float weighted sum uses 0.299, 0.587, 0.114; shape matches spatial dimensions.

**Failure mode**

Correct channel order or arithmetic overflow.

## 15. Compute Basic Image Statistics

**Purpose**

Summarize an image numerically.

**Inputs**

Loaded grayscale/RGB arrays.

**Procedure**

1. Select a grayscale image.
2. Compute min, max, mean, median, standard deviation, and selected percentiles.
3. Print them in a consistent summary.
4. Use percentiles to describe spread without relying only on extrema.

**Produces**

Minimum, maximum, mean and variability.

**Acceptance contract**

Statistics refer to the stated image and channel aggregation.

**Failure mode**

Investigate empty or nonfinite arrays before summarizing.

## 16. Compute and Validate Histograms

**Purpose**

Count intensities without losing pixels.

**Inputs**

8-bit grayscale image.

**Procedure**

1. Flatten a grayscale image and compute a 256-bin histogram over `[0,256)`.
2. Plot intensity versus pixel count.
3. Verify histogram counts sum exactly to the number of pixels.

**Produces**

256-bin histogram.

**Acceptance contract**

Sum of counts equals number of pixels; bins span all 8-bit values.

**Failure mode**

Correct bin range or grayscale conversion.

## 17. Show that Histograms Ignore Spatial Arrangement

**Purpose**

Demonstrate what a histogram cannot encode.

**Inputs**

An image and a pixel rearrangement.

**Procedure**

1. Copy and flatten the grayscale image.
2. Shuffle pixels with the fixed RNG and reshape to the original shape.
3. Compute histograms before and after shuffling.
4. Assert histograms are identical even though spatial structure is destroyed.

**Produces**

Equal histograms with different spatial layouts.

**Acceptance contract**

The multiset of pixel values is preserved.

**Failure mode**

Do not infer identical images from identical histograms.

## 18. Normalize Dynamic Range

**Purpose**

Expand observed intensity range with a constant-image case.

**Inputs**

Grayscale image.

**Procedure**

1. Implement min-max normalization in float arithmetic.
2. If maximum equals minimum, return a safe constant result.
3. Otherwise map the observed range to `[0,255]`, clip, and cast.
4. Compare image and histogram before/after normalization.

**Produces**

Min-max normalized uint8 image.

**Acceptance contract**

Nonconstant example spans 0 to 255; constant branch returns zeros.

**Failure mode**

Do not divide by zero or require full dynamic range from a constant input.

## 19. Demonstrate uint8 Arithmetic Safety

**Purpose**

Explain integer wraparound and safe arithmetic.

**Inputs**

uint8 pixels and a brightness offset.

**Procedure**

1. Create values near the uint8 limit.
2. Perform arithmetic directly in uint8 and observe wraparound.
3. Repeat in a wider or float type.
4. Clip and cast intentionally; reuse this safe pattern later.

**Produces**

Wrapped, floating and clipped examples.

**Acceptance contract**

Promote dtype before arithmetic; clip only the storage/display result.

**Failure mode**

Do not use wrapped values as a numerical enhancement result.

## 20. Simulate Common Noise Models

**Purpose**

Compare distinct corruption mechanisms.

**Inputs**

Clean image and seeded RNG.

**Procedure**

1. Use the fixed RNG to add zero-mean Gaussian noise.
2. Create salt-and-pepper corruption from a random probability map.
3. Create Poisson noise from scaled intensity-dependent counts.
4. Create multiplicative speckle noise.
5. Clip/cast final noisy images and save one common comparison.

**Produces**

Gaussian, impulse and speckle examples.

**Acceptance contract**

Same clean source; recorded parameters; final display values in range.

**Failure mode**

Check model/range errors; visually different samples are not comparable without common settings.

## 21. Implement Image-Quality Metrics

**Purpose**

Measure distortion against a stated reference.

**Inputs**

Two identically shaped images.

**Procedure**

1. Require reference and test arrays to have identical shapes.
2. Convert both to float64 and compute their difference.
3. Return MAE, MSE, RMSE, and PSNR.
4. Return infinite PSNR only when MSE is exactly zero.

**Produces**

MAE, MSE, RMSE and PSNR.

**Acceptance contract**

Self-comparison gives zero errors and infinite PSNR; arithmetic is floating point.

**Failure mode**

Raise on shape mismatch; infinite PSNR at zero MSE is valid.

## 22. Compare Lossless and Lossy Encoding

**Purpose**

Distinguish lossless pixels from compressed file size.

**Inputs**

One source saved as PNG and JPEG.

**Procedure**

1. Save the same RGB image as PNG and JPEG with explicit JPEG quality.
2. Reload both files as RGB arrays.
3. Compute quality metrics against the original.
4. Use the reloaded arrays so measured error includes real codec behaviour.

**Produces**

Reloaded images, files and comparisons.

**Acceptance contract**

Compare decoded pixels and byte sizes separately; use the same source.

**Failure mode**

Stop on I/O failure; do not assume smaller files preserve pixels exactly.

## 23. Implement a Standard Inspection Summary

**Purpose**

Reuse the same inspection vocabulary.

**Inputs**

An image and descriptive name.

**Procedure**

1. Create a helper printing image name, shape, dimensions, dtype, min/max, mean, and memory.
2. Call it on a representative image.
3. Keep this helper as the reusable inspection entry point for future datasets.

**Produces**

Property summary from summarize_image_properties.

**Acceptance contract**

Shape, dtype, range, mean and nbytes describe the passed array.

**Failure mode**

Fix a stale variable or mislabeled input before reporting.

## 24. Run Final Validation

**Purpose**

Check representation, counting and metric identities.

**Inputs**

Computed arrays, metrics and output paths.

**Procedure**

1. Verify expected RGB/grayscale dimensionality and uint8 types.
2. Verify histogram counts, normalized range, and zero self-error metrics.
3. Verify ROI geometry.
4. Verify all 16 declared output files exist.
5. Print the final pass message only after every check succeeds.

**Produces**

Final validation outcome.

**Acceptance contract**

Histogram count equals pixel count; ROI dimensions match; identity metrics and normalization checks pass; 16 artifacts exist.

**Failure mode**

Stop on failed assertions or missing artifacts; do not reuse stale files as evidence of a new run.